In [1]:
import os
import json
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from torch.amp import GradScaler
# Check GPU availability
if not torch.cuda.is_available():
    raise RuntimeError("This code requires a GPU to run! No GPU was detected.")

device = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# Load configuration
with open('./config.json') as f:
    config = json.load(f)

class GestureDataset(Dataset):
    def __init__(self, split):
        self.split = split
        data_config = config['data_config']
        self.csv_path = os.path.join(data_config['data_root'], f"{split}.csv")
        self.video_root = os.path.join(data_config['data_root'], split)
        self.sequence_length = config['data_config']['sequence_length']
        self.input_size = config['data_config']['input_size']

        # Initialize MediaPipe in the main process
        self.hands = mp.solutions.hands.Hands(
            static_image_mode=True,
            max_num_hands=1,
            min_detection_confidence=0.5
        )

        if not os.path.exists(self.csv_path):
            raise FileNotFoundError(f"Missing {self.csv_path}")

        df = pd.read_csv(self.csv_path, header=None, names=['data'])
        self.samples = []
        for _, row in df.iterrows():
            try:
                parts = row['data'].split(';')
                if len(parts) < 3:
                    continue
                folder_name = parts[0].strip()
                label = int(parts[-1])
                video_path = os.path.join(self.video_root, folder_name)
                if os.path.exists(video_path):
                    frames = sorted([f for f in os.listdir(video_path)
                                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                    if len(frames) >= 5:
                        self.samples.append((video_path, label))
            except Exception as e:
                print(f"Error processing row {_}: {str(e)}")
                continue

        print(f"Loaded {len(self.samples)} samples for {split}")

    def __len__(self):
        return len(self.samples)

    def process_frame(self, img_path):
        try:
            img = cv2.imread(img_path)
            if img is None:
                return np.zeros(21 * 3)

            img = cv2.resize(img, tuple(self.input_size))
            results = self.hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            if not results.multi_hand_landmarks:
                return np.zeros(21 * 3)

            hand = results.multi_hand_landmarks[0]
            return np.array([[lm.x, lm.y, lm.z] for lm in hand.landmark]).flatten()

        except Exception as e:
            print(f"Error processing frame {img_path}: {str(e)}")
            return np.zeros(21 * 3)

    def __getitem__(self, idx):
        try:
            video_path, label = self.samples[idx]
            sequence = []

            # Process each frame
            frame_files = sorted(os.listdir(video_path))[:self.sequence_length]
            for frame_file in frame_files:
                img_path = os.path.join(video_path, frame_file)
                landmarks = self.process_frame(img_path)
                sequence.append(landmarks)

            # Pad sequence if necessary
            if len(sequence) < self.sequence_length:
                padding = [np.zeros(21 * 3)] * (self.sequence_length - len(sequence))
                sequence.extend(padding)

            return torch.FloatTensor(np.array(sequence)), torch.tensor(label, dtype=torch.long)

        except Exception as e:
            print(f"Error loading sample {idx}: {str(e)}")
            return (torch.zeros(self.sequence_length, 21 * 3),
                    torch.tensor(0, dtype=torch.long))

    def __del__(self):
        if hasattr(self, 'hands'):
            self.hands.close()

def create_data_loaders(config):
    train_dataset = GestureDataset('train')
    val_dataset = GestureDataset('val')

    # Using zero workers to keep everything in the main process
    train_loader = DataLoader(
        train_dataset,
        batch_size=config['train_config']['batch_size'],
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config['train_config']['batch_size'],
        shuffle=False,
        num_workers=0
    )

    return train_loader, val_loader

class GestureRecognitionModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=21 * 3,
            hidden_size=config['model_config']['lstm_hidden_size'],
            num_layers=config['model_config']['num_layers'],
            bidirectional=config['model_config']['bidirectional'],
            dropout=config['model_config']['dropout'] if config['model_config']['num_layers'] > 1 else 0,
            batch_first=True
        )

        lstm_output_size = (config['model_config']['lstm_hidden_size'] *
                            (2 if config['model_config']['bidirectional'] else 1))

        self.fc = nn.Linear(lstm_output_size, config['model_config']['num_classes'])

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])

def train_model():
    try:
        model = GestureRecognitionModel(config).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=config['train_config']['learning_rate'])
        scaler = GradScaler('cuda')

        train_loader, val_loader = create_data_loaders(config)

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(config['train_config']['epochs']):
            # Training phase
            model.train()
            train_loss = 0

            for batch_idx, (data, target) in enumerate(train_loader):
                try:
                    data, target = data.to(device), target.to(device)

                    optimizer.zero_grad(set_to_none=True)

                    with autocast():
                        output = model(data)
                        loss = criterion(output, target)

                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()

                    train_loss += loss.item()

                    if batch_idx % 10 == 0:
                        print(f'Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}')

                except Exception as e:
                    print(f"Error in training batch {batch_idx}: {str(e)}")
                    continue

            # Validation phase
            model.eval()
            val_loss = 0
            correct = 0
            total = 0

            with torch.no_grad():
                for data, target in val_loader:
                    try:
                        data, target = data.to(device), target.to(device)
                        output = model(data)
                        val_loss += criterion(output, target).item()
                        pred = output.argmax(dim=1)
                        correct += pred.eq(target).sum().item()
                        total += target.size(0)
                    except Exception as e:
                        print(f"Error in validation batch: {str(e)}")
                        continue

            if total > 0:
                val_loss /= len(val_loader)
                accuracy = 100. * correct / total

                print(f'Epoch {epoch}: Val Loss: {val_loss:.4f}, Accuracy: {accuracy:.2f}%')

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': best_val_loss,
                    }, config['train_config']['checkpoint_path'])
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= config['train_config']['patience']:
                        print("Early stopping triggered")
                        break

            torch.cuda.empty_cache()

    except Exception as e:
        print(f"Training error: {str(e)}")
        raise

if __name__ == "__main__":
    try:
        train_model()
    except KeyboardInterrupt:
        print("Training interrupted by user")
    except Exception as e:
        print(f"Error in main: {str(e)}")

Using GPU: NVIDIA GeForce RTX 4050 Laptop GPU


libEGL warning: MESA-LOADER: failed to open radeonsi: /usr/lib/dri/radeonsi_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open radeonsi: /usr/lib/dri/radeonsi_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open radeonsi: /usr/lib/dri/radeonsi_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open radeonsi: /usr/lib/dri/radeonsi_dri.so: 

Loaded 943 samples for train
Loaded 280 samples for val


/tmp/ipykernel_4141/112283967.py:179: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 0, Batch 0, Loss: 1.7933
Epoch 0: Val Loss: 1.7409, Accuracy: 28.93%
Epoch 1, Batch 0, Loss: 1.7695
Epoch 1: Val Loss: 1.2636, Accuracy: 45.71%
Epoch 2, Batch 0, Loss: 1.3929
Epoch 2: Val Loss: 1.3173, Accuracy: 52.14%
Epoch 3, Batch 0, Loss: 1.3252
Epoch 3: Val Loss: 1.1372, Accuracy: 59.29%
Epoch 4, Batch 0, Loss: 1.1486
Epoch 4: Val Loss: 1.0039, Accuracy: 55.00%
Epoch 5, Batch 0, Loss: 1.1103
Epoch 5: Val Loss: 0.9622, Accuracy: 57.50%
Epoch 6, Batch 0, Loss: 0.8109
Epoch 6: Val Loss: 0.9875, Accuracy: 63.57%
Epoch 7, Batch 0, Loss: 0.8053
Epoch 7: Val Loss: 0.8979, Accuracy: 61.07%
Epoch 8, Batch 0, Loss: 0.6677
Epoch 8: Val Loss: 0.8143, Accuracy: 65.71%
Epoch 9, Batch 0, Loss: 0.7085
Epoch 9: Val Loss: 0.9616, Accuracy: 63.93%
Epoch 10, Batch 0, Loss: 0.6660
Epoch 10: Val Loss: 0.7640, Accuracy: 67.50%
Epoch 11, Batch 0, Loss: 0.6076
Epoch 11: Val Loss: 1.1190, Accuracy: 65.00%
Epoch 12, Batch 0, Loss: 0.7586
Epoch 12: Val Loss: 0.9906, Accuracy: 67.86%
Epoch 13, Batch 0, 